# Streaming a single catalog from Hugging Face

A minimal baseline: pull five chunks off one catalog with `InfiniteStream`, print what
comes back, and capture the worker side with dask's `performance_report`.

This is the **no-op consumer** case -- we do nothing with a chunk but measure it. That is
deliberate: with no consumer work to overlap, the time spent blocked on the workers is the
full worker-side cost, which makes this the reference point for a later run that actually
does something with each chunk.

In [1]:
from time import perf_counter

import pandas as pd
from dask.distributed import Client, performance_report, wait

import lsdb
from lsdb.streams import InfiniteStream

In [2]:
!mkdir -p reports/

## Cluster

Two worker processes, one thread each. Keep the dashboard **browser tab closed** while a
run is being timed -- a connected tab pushes websocket updates for the whole run and shows
up as event-loop churn. `performance_report` below collects the same diagnostics with no
browser attached.

In [3]:
client = Client(
    n_workers=2,
    threads_per_worker=1,
    memory_limit="32GB",
)
client

KeyError: 'JUPYTERHUB_PUBLIC_URL'

<Client: 'tcp://127.0.0.1:20599' processes=2 threads=2, memory=59.60 GiB>

## Catalog

`image.flux` -- a 3x152x152 float32 cutout stack per object -- is about 99.9% of the bytes
here, roughly 273 KB per row. Row counts are a poor throughput unit for this catalog, so
the summary below reports megabytes as well.

In [4]:
CATALOG = "hf://datasets/UniverseTBD/mmu_ssl_legacysurvey_north/"

cat = lsdb.open_catalog(CATALOG)
cat.npartitions

5488

## The run

`InfiniteStream.__next__` does three things in order: block on the pending future, shuffle
the chunk's rows, then submit the *next* chunk. The first submit happens back at `iter()`.

`dask.distributed.wait()` blocks on a future without consuming it, so bracketing it just
before `next()` separates the blocked-on-workers time from the main-thread work that
follows -- using only public attributes, with lsdb's source untouched. `next()` then runs
exactly as written; its internal `result()` simply returns straight away.

- `wait_s` -- blocked on the workers: HTTP fetch, parquet decode, `pd.concat`, serialization
- `post_s` -- shuffle plus submit, on this process's main thread

Chunk 0 is cold: it carries catalog metadata loading and connection setup, and had no
prefetch behind it. It is recorded, then excluded from the summary.

In [5]:
N_ITERS = 5
PARTITIONS_PER_CHUNK = 2
SEED = 1
REPORT_PATH = "reports/baseline-report.html"

stream = InfiniteStream(
    catalog=cat,
    client=client,
    partitions_per_chunk=PARTITIONS_PER_CHUNK,
    seed=SEED,
)

records = []

# The report renders on context exit, outside the per-chunk brackets, so it does not
# pollute the timings it wraps.
with performance_report(filename=REPORT_PATH):
    stream_iter = iter(stream)  # submits chunk 0

    for i in range(N_ITERS):
        t0 = perf_counter()
        wait(stream_iter.future)  # blocks without consuming the future
        t1 = perf_counter()
        chunk = next(stream_iter)  # shuffle, then submit chunk i+1
        t2 = perf_counter()

        # deep=True accounts the Arrow buffers behind the nested `image` column,
        # so this is real bytes rather than a pointer count.
        nbytes = int(chunk.memory_usage(deep=True).sum())
        records.append({"chunk": i, "rows": len(chunk), "nbytes": nbytes,
                        "wait_s": t1 - t0, "post_s": t2 - t1})

        print(f"chunk {i}: {len(chunk):>6} rows  {nbytes / 1e6:>7.1f} MB"
              f"  wait {t1 - t0:>6.2f}s  post {t2 - t1:>5.2f}s")

        del chunk  # drop it before the next lands; these are ~1.7 GB each

print(f"\nwrote {REPORT_PATH}")

chunk 0:   4423 rows   1235.3 MB  wait  23.15s  post  4.23s
chunk 1:   4379 rows   1223.0 MB  wait  20.99s  post  4.20s
chunk 2:  10479 rows   2926.6 MB  wait  44.72s  post 10.33s
chunk 3:   5556 rows   1551.7 MB  wait  23.03s  post  5.36s
chunk 4:   4596 rows   1283.6 MB  wait  17.85s  post  4.39s

wrote reports/baseline-report.html


## Summary

In [6]:
# Image floats barely compress (50.74 MB compressed against 56.62 MB uncompressed for one
# row group), so decoded bytes proxy bytes on the wire to about 11%.
WIRE_RATIO = 0.896

results = pd.DataFrame(records)
steady = results[results["chunk"] > 0]

mb = steady["nbytes"] / 1e6
wall_s = (steady["wait_s"] + steady["post_s"]).sum()

print(f"{len(steady)} steady chunks (chunk 0 excluded as cold)")
print(f"  {mb.sum():.0f} MB decoded in {wall_s:.1f}s")
print(f"  {wall_s / len(steady):.2f} s per chunk")
print(f"  wire   {WIRE_RATIO * mb.sum() / wall_s:.2f} MB/s end to end")
print(f"  wire   {WIRE_RATIO * mb.sum() / steady['wait_s'].sum():.2f} MB/s during fetch only")
print(f"  post_s {100 * steady['post_s'].sum() / wall_s:.1f}% of wall (workers idle for it)")

results

4 steady chunks (chunk 0 excluded as cold)
  6985 MB decoded in 130.9s
  32.72 s per chunk
  wire   47.82 MB/s end to end
  wire   58.71 MB/s during fetch only
  post_s 18.5% of wall (workers idle for it)


,chunk,rows,nbytes,wait_s,post_s
0,0,4423,1235256323,23.147811,4.231008
1,1,4379,1222968159,20.994076,4.199574
2,2,10479,2926577358,44.718858,10.331408
3,3,5556,1551680741,23.031365,5.358252
4,4,4596,1283571776,17.850458,4.386665
